In [1]:
import sys
print(sys.executable)

/workspaces/bot_ws/notebooks_venv/bin/python


In [2]:
import sympy as sp
sp.init_printing(use_unicode=True)
from IPython.display import Math, display

In [3]:
# Beam measurement index
i = sp.symbols('i', integer=True, nonnegative=True)

# From the sensor_msgs/msg/LaserScan 
# the ith range in the ranges array
rho_i = sp.symbols('rho_i', real=True)
# angle_min and angle_increment
theta_min = sp.symbols('theta_min', real=True)
Delta_theta = sp.Symbol(r'\Delta\theta', real=True)


In [4]:
# Intrinsic Parameters
b_rho, s_rho, b_theta = sp.symbols('b_rho s_rho b_theta', real=True)

In [5]:
# Extrinsic Parameters
c1, c2, c3 = sp.symbols('c1 c2 c3', real=True)
c = sp.Matrix([c1, c2, c3])

r11, r12, r13, r21, r22, r23, r31, r32, r33 = sp.symbols('r11 r21 r13 r21 r22 r23 r31 r32 r33', real=True)
R = sp.Matrix([
    [r11, r21, r13],
    [r21, r22, r23],
    [r31, r32, r33]
])

In [6]:
def skew(v):
    return sp.Matrix([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

In [7]:
theta_i = theta_min + i*Delta_theta + b_theta
q_i = s_rho*rho_i + b_rho

a_i = sp.Matrix([
    sp.cos(theta_i),
    sp.sin(theta_i),
    0
])

a_perp_i = sp.Matrix([
    -sp.sin(theta_i),
    sp.cos(theta_i),
    0
])

# Measurement in LiDAR frame
pL_i = q_i*a_i

# Measurement in the base_footprint frame
X_i = c + R.T*pL_i

display(Math(r'\theta_i = ' + sp.latex(theta_i)))
display(Math(r'q_i = ' + sp.latex(q_i)))
display(Math(r'X_i = ' + sp.latex(X_i)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
# Jacobian wrt the LiDAR Centre
J_c_i = X_i.jacobian(c)

In [9]:
# Jacobian wrt small angle perturbations in the 3-vector angle 
J_phi_i = R.T*skew(pL_i)

In [10]:
# Jacobian wrt intrinsic parameters
J_b_rho_i = X_i.diff(b_rho)
J_s_rho_i = X_i.diff(s_rho)
J_b_theta_i = X_i.diff(b_theta)

In [11]:
# Jacobian wrt the range measurement
J_rho_i = X_i.diff(rho_i)

In [12]:
# Anisotropic noise in the Camera Center
sigma_c_x, sigma_c_y, sigma_c_z = sp.symbols(
    r'\sigma_{c_1} \sigma_{c_2} \sigma_{c_3}',
    positive=True,
    real=True
)
Sigma_c = sp.diag(sigma_c_x**2, sigma_c_y**2, sigma_c_z**2)
display(Math(r'\Sigma_c = ' + sp.latex(Sigma_c)))

<IPython.core.display.Math object>

In [13]:
# Anisotropic noise in the angular perturbation
sigma_phi_x, sigma_phi_y, sigma_phi_z = sp.symbols(
    r'\sigma_{\phi_1} \sigma_{\phi_2} \sigma_{\phi_3}',
    positive=True,
    real=True
)
Sigma_phi = sp.diag(sigma_phi_x**2, sigma_phi_y**2, sigma_phi_z**2)
display(Math(r'\Sigma_{\phi} = ' + sp.latex(Sigma_phi)))

<IPython.core.display.Math object>

In [14]:
# noise in the intrinsic parameters
sigma_b_rho = sp.symbols(r'\sigma_{b_\rho}', positive=True, real=True)
sigma_s_rho = sp.symbols(r'\sigma_{s_\rho}', positive=True, real=True)
sigma_b_theta = sp.symbols(r'\sigma_{b_\theta}', positive=True, real=True)
display(sigma_b_rho, sigma_s_rho, sigma_b_theta)

In [15]:
# noise in the range measurement
sigma_rho_i = sp.symbols(r'\sigma_{\rho_i}', positive=True, real=True)
display(sigma_rho_i)

In [16]:
# Uncertainty in the 3D measurement
Sigma_X_i = (
    J_c_i*Sigma_c*J_c_i.T
    + J_phi_i*Sigma_phi*J_phi_i.T
    + (sigma_b_rho**2)*(J_b_rho_i*J_b_rho_i.T)
    + (sigma_s_rho**2)*(J_s_rho_i*J_s_rho_i.T)
    + (sigma_b_theta**2)*(J_b_theta_i*J_b_theta_i.T)
    + (sigma_rho_i**2)*(J_rho_i*J_rho_i.T)
)
display(Sigma_X_i)

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢               2                     2    2                                   ↪
⎢\sigma_{\phi_1} ⋅r₃₁⋅r₃₂⋅(bᵨ + ρᵢ⋅sᵨ) ⋅sin (\Delta\theta⋅i + bₜₕₑₜₐ + θₘᵢₙ) + ↪
⎢                                                                              ↪
⎢               2                     2    2                                   ↪
⎣\sigma_{\phi_1} ⋅r₃₁⋅r₃₃⋅(bᵨ + ρᵢ⋅sᵨ) ⋅sin (\Delta\theta⋅i + bₜₕₑₜₐ + θₘᵢₙ) + ↪

↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                 2                     2    2                                 ↪
↪  \sigma_{\phi_2} ⋅r₃₁⋅r₃₂

In [17]:
expr_test = X_i.subs({
    theta_min: 0,
    Delta_theta: 2*sp.pi/360,
    i: 10,
    rho_i: 2.5,
    s_rho: 1.0,
    b_rho: 0.0,
    b_theta: 0.0
})

sp.N(expr_test)

⎡c₁ + 2.46201938253052⋅r₁₁ + 0.434120444167326⋅r₂₁⎤
⎢                                                 ⎥
⎢c₂ + 2.46201938253052⋅r₂₁ + 0.434120444167326⋅r₂₂⎥
⎢                                                 ⎥
⎣c₃ + 2.46201938253052⋅r₁₃ + 0.434120444167326⋅r₂₃⎦